# Training ML-dSGP4

This tutorial shows the complete training/checkpoint workflow: prepare aligned TLE/time/reference-state samples, optimize `dsgp4.mldsgp4`, save the model `state_dict`, and reload it with `load_model`.

The reference states below are deliberately synthetic so the notebook is self-contained. For a real application, replace them with higher-precision propagated or observed **TEME** states in km and km/s.

In [ ]:
from pathlib import Path

import dsgp4
import torch

torch.manual_seed(7)

## Build a small training set

Each target state must correspond one-to-one with a TLE and a `tsince` value. The model output is normalized, so the reference position and velocity are normalized with the model constants before computing the loss.

In [ ]:
tles = dsgp4.tle.load("example.tle")
samples_per_tle = 2

training_tles = []
training_times = []
for tle in tles:
    training_tles.extend([tle.copy() for _ in range(samples_per_tle)])
    training_times.append(torch.linspace(0.0, 120.0, samples_per_tle))

tsinces = torch.cat(training_times)

# Create self-contained reference data from dSGP4 and add a small deterministic
# correction. Replace `reference_states` with higher-precision/observed TEME
# states for real training. Expected shape: [N, 2, 3], in km and km/s.
with torch.no_grad():
    reference_states = dsgp4.propagate_batch(
        training_tles, tsinces, initialized=False
    ).clone()
    reference_states[:, 0, :] *= 1.00005
    reference_states[:, 1, :] *= 0.99995

model = dsgp4.mldsgp4(hidden_size=8)
target = torch.cat(
    (
        reference_states[:, 0, :] / model.normalization_R,
        reference_states[:, 1, :] / model.normalization_V,
    ),
    dim=1,
)

target.shape

## Optimize the model

ML-dSGP4 is a regular PyTorch module, so standard optimizers and losses work. This short loop is intentionally small so the documentation remains quick to execute; real training normally uses more data, a validation split, and task-specific optimization settings.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.MSELoss()
losses = []

model.train()
for _ in range(10):
    optimizer.zero_grad()
    prediction = model(training_tles, tsinces)
    loss = criterion(prediction, target)
    loss.backward()
    optimizer.step()
    losses.append(loss.detach().item())

print(f"initial loss: {losses[0]:.6e}")
print(f"final loss:   {losses[-1]:.6e}")

## Save and reload

Save only the PyTorch `state_dict`. When loading, create `mldsgp4` with the same architecture (for example the same `hidden_size` and normalization constants), then call `load_model`.

In [ ]:
checkpoint = Path("mldsgp4_training_example.pth")
torch.save(model.state_dict(), checkpoint)

model.eval()
with torch.no_grad():
    expected = model(training_tles, tsinces)

restored = dsgp4.mldsgp4(hidden_size=8)
restored.load_model(checkpoint, device="cpu")
with torch.no_grad():
    actual = restored(training_tles, tsinces)

torch.testing.assert_close(actual, expected)
checkpoint.unlink()
print("checkpoint reload matches the trained model")

## Using real reference data

For operational training, replace the synthetic `reference_states` with states from a higher-precision numerical propagator or observations transformed to TEME. Keep the sample order aligned with `training_tles` and `tsinces`, preserve km/km/s units before normalization, and evaluate on held-out satellites or time intervals rather than only on the training samples.

The existing `mldsgp4.ipynb` tutorial shows inference with the repository's example pre-trained checkpoint.